In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

import brain_probability_flux.plotting as bpp

In [ ]:
# Results are read from data/. To plot a fresh run, set DATA_DIR to its run
# directory, for example Path("../data/2026-10-20").
DATA_DIR = bpp.DATA_DIR

In [ ]:
max_fig_width = bpp.FIG_WIDTH
max_fig_height = bpp.FIG_HEIGHT
panel_gap = bpp.PANEL_GAP_PT / 72  # gap between panels in the composite (in)

TASK_NAMES = bpp.TASK_NAMES_SHORT
NUM_TASKS = 8
NUM_SPINS = 7
NS = 2**NUM_SPINS
N_BOOT = 200  # trajectory bootstrap resamples

OUT_DIR = Path("../figures/fig-s09")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
pfa, cluster_inds = bpp.aggregated_pfa(data_dir=DATA_DIR)

In [ ]:
arr = bpp.load_results("inferred_networks", "arrhenius", DATA_DIR)
gla = bpp.load_results("inferred_networks", "glauber", DATA_DIR)
boot_arr = bpp.load_results("inference_bootstrap", "arrhenius", DATA_DIR)
boot_gla = bpp.load_results("inference_bootstrap", "glauber", DATA_DIR)
# Resampled state distributions (scripts/bootstrap_stationary.py)
probability_boot = bpp.load_results("stationary_bootstrap", data_dir=DATA_DIR)[
    "probability_boot"
]

beta_J_arr = arr["beta_J"]  # (8, 7, 7)
beta_h_arr = arr["beta_h"]  # (8, 7)
A_arr = arr["rate_const"]  # (8,)
beta_J_gla = gla["beta_J"]
beta_h_gla = gla["beta_h"]
A_gla = gla["rate_const"]

bJ_arr_boot = boot_arr["beta_J"]  # (8, 100, 7, 7)
bJ_gla_boot = boot_gla["beta_J"]  # (8, 100, 7, 7)
N_BOOT_PARAM = bJ_arr_boot.shape[1]  # 100

In [ ]:
SV = bpp.create_state_vectors(
    NUM_SPINS
)  # (128, 7)  spin vectors for all states

In [ ]:
p_emp_all = []
p_arr_all = []
p_gla_all = []
r_arr = []
r_gla = []
jsd_arr = []
jsd_gla = []
ref_arr_all = []  # reference fields for Arrhenius fitting loss
ref_gla_all = []  # reference fields for Glauber fitting loss (masked w < A)

for k in range(NUM_TASKS):
    pfa.trajectories_to_flux(k, progress_message=False)
    p_emp = pfa.probability_task.copy()

    W_a = bpp.build_W_arrhenius(
        beta_J_arr[k], beta_h_arr[k], float(A_arr[k]), SV
    )
    W_g = bpp.build_W_glauber(
        beta_J_gla[k], beta_h_gla[k], float(A_gla[k]), SV
    )
    p_a = bpp.stationary_distribution(W_a)
    p_g = bpp.stationary_distribution(W_g)

    p_emp_all.append(p_emp)
    p_arr_all.append(p_a)
    p_gla_all.append(p_g)
    r_arr.append(stats.pearsonr(p_emp, p_a).statistic)
    r_gla.append(stats.pearsonr(p_emp, p_g).statistic)
    jsd_arr.append(bpp.jsd(p_emp, p_a))
    jsd_gla.append(bpp.jsd(p_emp, p_g))

    # reference fields for fitting loss (panel f): precompute once per task
    _tr = pfa.transition_rate_matrix()
    _mu_list, _ki_list, _w_list, _sk_list = [], [], [], []
    for _mu in range(NS):
        for _ki in range(NUM_SPINS):
            _nu = _mu ^ (1 << (NUM_SPINS - 1 - _ki))
            _wv = _tr[_nu, _mu]
            if _wv <= 0:
                continue
            _mu_list.append(_mu)
            _ki_list.append(_ki)
            _w_list.append(_wv)
            _sk_list.append(SV[_mu, _ki])
    _mu_a = np.array(_mu_list)
    _ki_a = np.array(_ki_list)
    _w_a = np.array(_w_list)
    _sk_a = np.array(_sk_list)
    _Aa = float(A_arr[k])
    _Ag = float(A_gla[k])

    # Arrhenius reference field: beta*g_k = -s_k * ln(w/A) (no masking needed)
    _ref_a = -_sk_a * np.log(_w_a / _Aa)
    ref_arr_all.append((_mu_a, _ki_a, _ref_a))

    # Glauber reference field: beta*g_k = (s_k/2) * ln(A/w - 1), feasible only for w < A
    _fg = _w_a < _Ag
    _ref_g = _sk_a[_fg] / 2 * np.log(_Ag / _w_a[_fg] - 1)
    ref_gla_all.append((_mu_a[_fg], _ki_a[_fg], _ref_g))

print(
    f"{'Task':14s}  {'r (Arr)':>8s}  {'JSD (Arr)':>10s}  "
    f"{'r (Gla)':>8s}  {'JSD (Gla)':>10s}"
)
for k in range(NUM_TASKS):
    print(
        f"{TASK_NAMES[k]:14s}  {r_arr[k]:8.4f}  {jsd_arr[k]:10.5f}  "
        f"{r_gla[k]:8.4f}  {jsd_gla[k]:10.5f}"
    )

In [ ]:
def _eval_loss(bJ, bh, mu_idx, ki_idx, ref):
    g = np.einsum("nj,nj->n", bJ[ki_idx], SV[mu_idx]) + bh[ki_idx]
    return float(np.mean((g - ref) ** 2))


jsd_eq_arr = []
jsd_eq_gla = []
loss_fit_arr = []
loss_fit_gla = []

for k in range(NUM_TASKS):
    pi_arr = bpp.boltzmann_eq(beta_J_arr[k], beta_h_arr[k], SV)
    pi_gla = bpp.boltzmann_eq(beta_J_gla[k], beta_h_gla[k], SV)
    jsd_eq_arr.append(bpp.jsd(p_arr_all[k], pi_arr))
    jsd_eq_gla.append(bpp.jsd(p_gla_all[k], pi_gla))

    mu_a, ki_a, ref_a = ref_arr_all[k]
    mu_g, ki_g, ref_g = ref_gla_all[k]
    loss_fit_arr.append(
        _eval_loss(beta_J_arr[k], beta_h_arr[k], mu_a, ki_a, ref_a)
    )
    loss_fit_gla.append(
        _eval_loss(beta_J_gla[k], beta_h_gla[k], mu_g, ki_g, ref_g)
    )

print(
    f"{'Task':14s}  {'JSD_eq(Arr)':>12s}  {'JSD_eq(Gla)':>12s}  "
    f"{'Loss(Arr)':>10s}  {'Loss(Gla)':>10s}"
)
for k in range(NUM_TASKS):
    print(
        f"{TASK_NAMES[k]:14s}  {jsd_eq_arr[k]:12.6f}  {jsd_eq_gla[k]:12.6f}  "
        f"{loss_fit_arr[k]:10.6f}  {loss_fit_gla[k]:10.6f}"
    )

In [ ]:
jsd_boot_arr = np.zeros((NUM_TASKS, N_BOOT))
jsd_boot_gla = np.zeros((NUM_TASKS, N_BOOT))
r_boot_arr = np.zeros((NUM_TASKS, N_BOOT))
r_boot_gla = np.zeros((NUM_TASKS, N_BOOT))

for k in range(NUM_TASKS):
    for b in range(N_BOOT):
        p_boot = probability_boot[k, b]
        jsd_boot_arr[k, b] = bpp.jsd(p_boot, p_arr_all[k])
        jsd_boot_gla[k, b] = bpp.jsd(p_boot, p_gla_all[k])
        r_boot_arr[k, b] = stats.pearsonr(p_boot, p_arr_all[k]).statistic
        r_boot_gla[k, b] = stats.pearsonr(p_boot, p_gla_all[k]).statistic

jsd_arr_q25 = np.percentile(jsd_boot_arr, 25, axis=1)
jsd_arr_q75 = np.percentile(jsd_boot_arr, 75, axis=1)
jsd_gla_q25 = np.percentile(jsd_boot_gla, 25, axis=1)
jsd_gla_q75 = np.percentile(jsd_boot_gla, 75, axis=1)
jsd_arr_med = np.median(jsd_boot_arr, axis=1)
jsd_gla_med = np.median(jsd_boot_gla, axis=1)

r_arr_q25 = np.percentile(r_boot_arr, 25, axis=1)
r_arr_q75 = np.percentile(r_boot_arr, 75, axis=1)
r_gla_q25 = np.percentile(r_boot_gla, 25, axis=1)
r_gla_q75 = np.percentile(r_boot_gla, 75, axis=1)
r_arr_med = np.median(r_boot_arr, axis=1)
r_gla_med = np.median(r_boot_gla, axis=1)
print("Trajectory bootstrap done")

In [ ]:
jsd_eq_arr_boot = np.zeros((NUM_TASKS, N_BOOT_PARAM))
jsd_eq_gla_boot = np.zeros((NUM_TASKS, N_BOOT_PARAM))
loss_fit_arr_boot = np.zeros((NUM_TASKS, N_BOOT_PARAM))
loss_fit_gla_boot = np.zeros((NUM_TASKS, N_BOOT_PARAM))

for k in range(NUM_TASKS):
    mu_a, ki_a, ref_a = ref_arr_all[k]
    mu_g, ki_g, ref_g = ref_gla_all[k]
    for b in range(N_BOOT_PARAM):
        bJ_a_b = bJ_arr_boot[k, b]
        bJ_g_b = bJ_gla_boot[k, b]

        W_a_b = bpp.build_W_arrhenius(
            bJ_a_b, beta_h_arr[k], float(A_arr[k]), SV
        )
        W_g_b = bpp.build_W_glauber(bJ_g_b, beta_h_gla[k], float(A_gla[k]), SV)
        p_a_b = bpp.stationary_distribution(W_a_b)
        p_g_b = bpp.stationary_distribution(W_g_b)
        pi_a_b = bpp.boltzmann_eq(bJ_a_b, beta_h_arr[k], SV)
        pi_g_b = bpp.boltzmann_eq(bJ_g_b, beta_h_gla[k], SV)
        jsd_eq_arr_boot[k, b] = bpp.jsd(p_a_b, pi_a_b)
        jsd_eq_gla_boot[k, b] = bpp.jsd(p_g_b, pi_g_b)

        loss_fit_arr_boot[k, b] = _eval_loss(
            bJ_a_b, beta_h_arr[k], mu_a, ki_a, ref_a
        )
        loss_fit_gla_boot[k, b] = _eval_loss(
            bJ_g_b, beta_h_gla[k], mu_g, ki_g, ref_g
        )

jsd_eq_arr_q25 = np.percentile(jsd_eq_arr_boot, 25, axis=1)
jsd_eq_arr_q75 = np.percentile(jsd_eq_arr_boot, 75, axis=1)
jsd_eq_gla_q25 = np.percentile(jsd_eq_gla_boot, 25, axis=1)
jsd_eq_gla_q75 = np.percentile(jsd_eq_gla_boot, 75, axis=1)
jsd_eq_arr_med = np.median(jsd_eq_arr_boot, axis=1)
jsd_eq_gla_med = np.median(jsd_eq_gla_boot, axis=1)

loss_fit_arr_q25 = np.percentile(loss_fit_arr_boot, 25, axis=1)
loss_fit_arr_q75 = np.percentile(loss_fit_arr_boot, 75, axis=1)
loss_fit_gla_q25 = np.percentile(loss_fit_gla_boot, 25, axis=1)
loss_fit_gla_q75 = np.percentile(loss_fit_gla_boot, 75, axis=1)
loss_fit_arr_med = np.median(loss_fit_arr_boot, axis=1)
loss_fit_gla_med = np.median(loss_fit_gla_boot, axis=1)
print("Parameter bootstrap done")

In [ ]:
task_colors = bpp.task_colors_linspace(NUM_TASKS)
x = np.arange(NUM_TASKS)
axlw = plt.rcParams["axes.linewidth"]

all_vals = np.concatenate(p_emp_all + p_arr_all + p_gla_all)
shared_lim = float(all_vals.max()) * 1.05

r_pooled_arr = stats.pearsonr(
    np.concatenate(p_emp_all), np.concatenate(p_arr_all)
).statistic
r_pooled_gla = stats.pearsonr(
    np.concatenate(p_emp_all), np.concatenate(p_gla_all)
).statistic

fig, axs = plt.subplots(
    2, 3, figsize=(max_fig_width - 2 * panel_gap, max_fig_height * 2 / 3)
)

# ── Panel a: p_model (Arrhenius) vs p_data ──────────────────────────────────
ax = axs[0, 0]
for k in range(NUM_TASKS):
    ax.scatter(
        p_emp_all[k],
        p_arr_all[k],
        s=4,
        alpha=0.5,
        color=task_colors[k],
        label=TASK_NAMES[k],
        edgecolors="none",
        rasterized=True,
    )
ax.plot([0, shared_lim], [0, shared_lim], "k--", lw=axlw)
ax.set_xlim(0, shared_lim)
ax.set_ylim(0, shared_lim)
ax.set_xlabel(r"$p_{\mathrm{data}}$")
ax.set_ylabel(r"Arrhenius $p_{\mathrm{model}}$")
ax.legend(markerscale=2, frameon=False, ncol=1, loc="lower right", fontsize=5)
ax.text(
    0.02,
    0.98,
    rf"$r = {r_pooled_arr:.3f}$",
    transform=ax.transAxes,
    va="top",
    ha="left",
)

# ── Panel b: p_model (Glauber) vs p_data ────────────────────────────────────
ax = axs[0, 1]
for k in range(NUM_TASKS):
    ax.scatter(
        p_emp_all[k],
        p_gla_all[k],
        s=4,
        alpha=0.5,
        color=task_colors[k],
        edgecolors="none",
        rasterized=True,
    )
ax.plot([0, shared_lim], [0, shared_lim], "k--", lw=axlw)
ax.set_xlim(0, shared_lim)
ax.set_ylim(0, shared_lim)
ax.set_xlabel(r"$p_{\mathrm{data}}$")
ax.set_ylabel(r"Glauber $p_{\mathrm{model}}$")
ax.text(
    0.02,
    0.98,
    rf"$r = {r_pooled_gla:.3f}$",
    transform=ax.transAxes,
    va="top",
    ha="left",
)

# ── Panel c: Pearson r with IQR error bars ───────────────────────────────────
ax = axs[0, 2]
err_r_arr = np.array([r_arr_med - r_arr_q25, r_arr_q75 - r_arr_med])
err_r_gla = np.array([r_gla_med - r_gla_q25, r_gla_q75 - r_gla_med])
ax.bar(
    x - 0.2,
    r_arr_med,
    0.4,
    color="0.4",
    label="Arrhenius",
    yerr=err_r_arr,
    capsize=2,
    error_kw={"elinewidth": axlw},
)
ax.bar(
    x + 0.2,
    r_gla_med,
    0.4,
    color="0.75",
    label="Glauber",
    yerr=err_r_gla,
    capsize=2,
    error_kw={"elinewidth": axlw},
)
ax.set_xticks(x)
ax.set_xticklabels(TASK_NAMES, rotation=90)
ax.set_ylabel(r"Pearson $r(p_{\mathrm{model}}, p_{\mathrm{data}})$")
r_min = min(r_arr_med.min(), r_gla_med.min())
ax.set_ylim(max(0.0, r_min - 0.02), 1.0)
ax.legend(frameon=False)

# ── Panel d: D_JS(p_model || p_data) ────────────────────────────────────────
ax = axs[1, 0]
err_d_arr = np.array([jsd_arr_med - jsd_arr_q25, jsd_arr_q75 - jsd_arr_med])
err_d_gla = np.array([jsd_gla_med - jsd_gla_q25, jsd_gla_q75 - jsd_gla_med])
ax.bar(
    x - 0.2,
    jsd_arr_med,
    0.4,
    color="0.4",
    label="Arrhenius",
    yerr=err_d_arr,
    capsize=2,
    error_kw={"elinewidth": axlw},
)
ax.bar(
    x + 0.2,
    jsd_gla_med,
    0.4,
    color="0.75",
    label="Glauber",
    yerr=err_d_gla,
    capsize=2,
    error_kw={"elinewidth": axlw},
)
ax.set_xticks(x)
ax.set_xticklabels(TASK_NAMES, rotation=90)
ax.set_ylabel(
    r"$D_{\mathrm{JS}}(p_{\mathrm{model}}\,\|\,p_{\mathrm{data}})$ (nats)"
)
ax.set_ylim(0, 0.040)
ax.legend(frameon=False)

# ── Panel e: D_JS(p_model || pi) ─────────────────────────────────────────────
ax = axs[1, 1]
err_e_arr = np.array(
    [jsd_eq_arr_med - jsd_eq_arr_q25, jsd_eq_arr_q75 - jsd_eq_arr_med]
)
err_e_gla = np.array(
    [jsd_eq_gla_med - jsd_eq_gla_q25, jsd_eq_gla_q75 - jsd_eq_gla_med]
)
ax.bar(
    x - 0.2,
    jsd_eq_arr_med,
    0.4,
    color="0.4",
    label="Arrhenius",
    yerr=err_e_arr,
    capsize=2,
    error_kw={"elinewidth": axlw},
)
ax.bar(
    x + 0.2,
    jsd_eq_gla_med,
    0.4,
    color="0.75",
    label="Glauber",
    yerr=err_e_gla,
    capsize=2,
    error_kw={"elinewidth": axlw},
)
ax.set_xticks(x)
ax.set_xticklabels(TASK_NAMES, rotation=90)
ax.set_ylabel(r"$D_{\mathrm{JS}}(p_{\mathrm{model}}\,\|\,\pi)$ (nats)")
ax.legend(frameon=False)

# ── Panel f: Fitting loss ────────────────────────────────────────────────────
ax = axs[1, 2]

err_f_arr = np.array(
    [loss_fit_arr_med - loss_fit_arr_q25, loss_fit_arr_q75 - loss_fit_arr_med]
)
err_f_gla = np.array(
    [loss_fit_gla_med - loss_fit_gla_q25, loss_fit_gla_q75 - loss_fit_gla_med]
)

ax.bar(
    x - 0.2,
    loss_fit_arr_med,
    0.4,
    color="0.4",
    yerr=err_f_arr,
    capsize=2,
    error_kw={"elinewidth": axlw},
    label="Arrhenius",
)
ax.bar(
    x + 0.2,
    loss_fit_gla_med,
    0.4,
    color="0.75",
    yerr=err_f_gla,
    capsize=2,
    error_kw={"elinewidth": axlw},
    label="Glauber",
)

ax.set_xticks(x)
ax.set_xticklabels(TASK_NAMES, rotation=90)
ax.set_ylabel(r"Loss $\langle(\beta g_k - g_k^{\mathrm{ref}})^2\rangle$")
ax.legend(frameon=False)

bpp.save_axes_panels(fig, dict(zip("abcdef", axs.flat)), OUT_DIR)